# Assignment - Books API with Flask

## Part A - Code

In [1]:
from flask import Flask, request, jsonify
from bson import ObjectId
import json

app = Flask(__name__)

books = [{'id': 1, 'title': 'Clean Code', 'author': 'Robert C. Martin', "year": 2008},
        {'id': 2, "title": "Fluent Python", "author": "Luciano Ramalho", "year":2015}]
next_id = 3

# Helper
class JSONEncoder(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, ObjectId):
            return str(o)
        return super().default(o)
    
def make_response(data, status=200):
    return jsonify(json.loads(json.dumps(data, cls=JSONEncoder))), status

In [2]:
# Task 1
def find_book(book_id):
    for book in books:
        if book['id'] == book_id:
            return (book)
    return None

In [3]:
# Check Task 1
assert find_book(1)is not None, "find_book(1) returned None - book 1 exists"
assert find_book(1)["title"] == "Clean Code"
assert find_book(2)["author"] == "Luciano Ramalho"
assert find_book(99) is None
print("Task 1 passed")

Task 1 passed


In [4]:
# Task 2
def validate_book(data):
    # return an error message (str) when a field is missing or year is not an int
    for field in ('title','author','year'):
        if field not in data:
            return f"no {field}"
    if not isinstance(data['year'], int):
        return 'year must be an int'
    # return None when the data is valid
    return None

In [5]:
# Check Task 2
assert validate_book({"title":"A","author":"B","year":2000}) is None
assert isinstance(validate_book({"title":"A","author":"B"}),str) # no year
assert isinstance(validate_book({"title":"A","year":2000}),str) # no author
assert isinstance(validate_book({"author":"B","year":2000}),str) # no title
assert isinstance(validate_book({"title":"A","author":"B","year":"2000"}),str) # year is text
print("Task 2 passed")

Task 2 passed


In [6]:
# Task 3
@app.route('/books', methods=['GET'])
def list_books():
    return make_response(books)

In [7]:
# Task 4
@app.route('/books/<int:book_id>', methods=['GET'])
def get_book(book_id):
    book = find_book(book_id)
    if book is None:
        return make_response({"error": "Book not found"}, 404)
    return make_response(book)

In [8]:
# Task 5
@app.route('/books', methods=['POST'])
def add_book():
    global next_id
    data = request.get_json(silent=True) or {}
    err = validate_book(data)
    if err:
        return make_response({"error": err}, 400)
    book = {"id": next_id, "title": data["title"],
            "author": data["author"], "year": data["year"]}
    next_id += 1
    books.append(book)
    return make_response(book, 201)

In [9]:
# Task 6
@app.route('/books/<int:book_id>', methods=['PUT'])
def replace_book(book_id):
    data = request.get_json(silent=True) or {}
    err = validate_book(data)
    if err:
        return make_response({"error": err}, 400)
    book = find_book(book_id)
    if book is None:
        return make_response({"error": "Book not found"}, 404)
    book.update(title=data["title"], author=data["author"], year=data["year"])
    return make_response(book)

In [10]:
# Task 7
@app.route('/books/<int:book_id>', methods=['PATCH'])
def update_book(book_id):
    data = request.get_json(silent=True) or {}
    book = find_book(book_id)
    if book is None:
        return make_response({"error": "Book not found"}, 404)
    if "year" in data and not isinstance(data["year"], int):
        return make_response({"error": "year must be an int"}, 400)
    for field in ("title", "author", "year"):
        if field in data:
            book[field] = data[field]
    return make_response(book)

In [11]:
# Task 8
@app.route('/books/<int:book_id>', methods=['DELETE'])
def delete_book(book_id):
    book = find_book(book_id)
    if book is None:
        return make_response({"error": "Book not found"}, 404)
    books.remove(book)
    return "", 204

In [12]:
from threading import Thread 
from werkzeug.serving import make_server
server = make_server("127.0.0.1", 5001,app)
Thread(target=server.serve_forever, daemon=True).start()
print("Server running on http://127.0.0.1:5001")

Server running on http://127.0.0.1:5001


### Check endpoints

In [13]:
import requests
BASE="http://127.0.0.1:5001"
def show(r):
    """Print the status code and the body - also when the body is not JSON (e.g. Flask's HTML 404/500 page)."""
    try:
        body = r.json()
    except ValueError:
        body = r.text[:80]
    print(r.status_code,body)

In [14]:
# Check Task 3 - GET /books
r = requests.get(f"{BASE}/books")
show(r)

assert r.status_code == 200
assert [b["title"]for b in r.json()] == ["Clean Code", "Fluent Python"]
print("Task 3 passed")

127.0.0.1 - - [11/Oct/2026 20:01:46] "GET /books HTTP/1.1" 200 -


200 [{'author': 'Robert C. Martin', 'id': 1, 'title': 'Clean Code', 'year': 2008}, {'author': 'Luciano Ramalho', 'id': 2, 'title': 'Fluent Python', 'year': 2015}]
Task 3 passed


In [15]:
# Check Task 4 - GET /books/<book_id>
r = requests.get(f"{BASE}/books/2")
show(r)
assert r.status_code == 200
assert r.json() == {"id":2,"title":"Fluent Python", "author":"Luciano Ramalho", "year":2015}

r = requests.get(f"{BASE}/books/99")
show(r)
assert r.status_code == 404
assert r.json() == {"error":"Book not found"}

r=requests.get(f"{BASE}/books/abc") # not an int -> no route matches
assert r.status_code == 404
print("Task 4 passed")

127.0.0.1 - - [11/Oct/2026 20:01:46] "GET /books/2 HTTP/1.1" 200 -
127.0.0.1 - - [11/Oct/2026 20:01:46] "GET /books/99 HTTP/1.1" 404 -
127.0.0.1 - - [11/Oct/2026 20:01:46] "GET /books/abc HTTP/1.1" 404 -


200 {'author': 'Luciano Ramalho', 'id': 2, 'title': 'Fluent Python', 'year': 2015}
404 {'error': 'Book not found'}
Task 4 passed


In [16]:
# Check Task 5 - POST /books
r = requests.post(f"{BASE}/books",json = {"title":"Automate the Boring Stuff", "author":"Al Sweigart", "year":2019})
show(r)
assert r.status_code == 201
assert r.json() == {"id":3, "title":"Automate the Boring Stuff", "author":"Al Sweigart", "year":2019}


r = requests.post(f"{BASE}/books",json = {"title":"No Year", "author":"Someone"}) # missing year
show(r)
assert r.status_code == 400
assert "error"in r.json()
r = requests.post(f"{BASE}/books",json = {"title":"Text Year", "author":"Someone",
                                        "year":"2019"}) # year is text
show(r)
assert r.status_code == 400

r = requests.get(f"{BASE}/books")
assert len(r.json()) == 3, "the two bad requests must NOT add a book"
print("Task 5 passed")

127.0.0.1 - - [11/Oct/2026 20:01:46] "POST /books HTTP/1.1" 201 -
127.0.0.1 - - [11/Oct/2026 20:01:46] "POST /books HTTP/1.1" 400 -
127.0.0.1 - - [11/Oct/2026 20:01:46] "POST /books HTTP/1.1" 400 -
127.0.0.1 - - [11/Oct/2026 20:01:46] "GET /books HTTP/1.1" 200 -


201 {'author': 'Al Sweigart', 'id': 3, 'title': 'Automate the Boring Stuff', 'year': 2019}
400 {'error': 'no year'}
400 {'error': 'year must be an int'}
Task 5 passed


In [17]:
# Check Task 6 - PUT /books/<book_id>
new_data = {"title":"Automate the Boring Stuff with Python", "author":"Al Sweigart", "year":2020}
r = requests.put(f"{BASE}/books/3", json=new_data)
show(r)
assert r.status_code == 200
assert r.json() == {"id":3,**new_data} 

r = requests.put(f"{BASE}/books/3", json={"year":2021}) # incomplete -> refused
show(r)
assert r.status_code==400
assert requests.get(f"{BASE}/books/3").json()["year"]==2020, "a refused PUT must not change the book"

r = requests.put(f"{BASE}/books/99", json=new_data)
show(r)
assert r.status_code == 404
print("Task 6 passed")

127.0.0.1 - - [11/Oct/2026 20:01:46] "PUT /books/3 HTTP/1.1" 200 -
127.0.0.1 - - [11/Oct/2026 20:01:46] "PUT /books/3 HTTP/1.1" 400 -
127.0.0.1 - - [11/Oct/2026 20:01:46] "GET /books/3 HTTP/1.1" 200 -
127.0.0.1 - - [11/Oct/2026 20:01:46] "PUT /books/99 HTTP/1.1

200 {'author': 'Al Sweigart', 'id': 3, 'title': 'Automate the Boring Stuff with Python', 'year': 2020}
400 {'error': 'no title'}


" 404 -


404 {'error': 'Book not found'}
Task 6 passed


In [18]:
r = requests.get("http://localhost:5001/books/3")
print(r.status_code)
print(r.text)
print(r.json())

127.0.0.1 - - [11/Oct/2026 20:01:48] "GET /books/3 HTTP/1.1" 200 -


200
{"author":"Al Sweigart","id":3,"title":"Automate the Boring Stuff with Python","year":2020}

{'author': 'Al Sweigart', 'id': 3, 'title': 'Automate the Boring Stuff with Python', 'year': 2020}


In [19]:
# Check Task 7 - PATCH /books/<book_id>
r = requests.patch(f"{BASE}/books/3", json={"year": 2021})        # only the year
show(r)
assert r.status_code == 200
assert r.json() == {"id": 3, "title": "Automate the Boring Stuff with Python", "author": "Al Sweigart", "year": 2021}

r = requests.patch(f"{BASE}/books/3", json={"id": 50, "title": "ATBS"})   # id must be ignored
show(r)
assert r.status_code == 200
assert r.json()["id"] == 3
assert r.json()["title"] == "ATBS"

r = requests.patch(f"{BASE}/books/3", json={"title": "Changed", "year": "new"})   # bad year
show(r)
assert r.status_code == 400
assert requests.get(f"{BASE}/books/3").json()["title"] == "ATBS", "a refused PATCH must notchange anything"

r = requests.patch(f"{BASE}/books/99", json={"year": 2000})
assert r.status_code == 404
print("Task 7 passed")


127.0.0.1 - - [11/Oct/2026 20:01:48] "PATCH /books/3 HTTP/1.1" 200 -
127.0.0.1 - - [11/Oct/2026 20:01:48] "PATCH /books/3 HTTP/1.1" 200 -
127.0.0.1 - - [11/Oct/2026 20:01:48] "PATCH /books/3 HTTP/1.1" 400 -
127.0.0.1 - - [11/Oct/2026 20:01:48] "GET /books/3 HTTP/1.1" 200 -
127.0.0.1 - - [11/Oct/2026 20:01:48] "PATCH /books/99 HTTP/1.1" 404 -


200 {'author': 'Al Sweigart', 'id': 3, 'title': 'Automate the Boring Stuff with Python', 'year': 2021}
200 {'author': 'Al Sweigart', 'id': 3, 'title': 'ATBS', 'year': 2021}
400 {'error': 'year must be an int'}
Task 7 passed


In [20]:
# Check Task 8 - DELETE /books/<book_id>
r = requests.delete(f"{BASE}/books/1")
print(r.status_code, repr(r.text))
assert r.status_code == 204
assert r.text == ""
assert requests.get(f"{BASE}/books/1").status_code == 404     # it is gone

r = requests.delete(f"{BASE}/books/1")                         # delete it again
show(r)
assert r.status_code == 404
# The id problem from the lesson: with len(books) + 1 this new book would get id 3 - alreadytaken!

r = requests.post(f"{BASE}/books", json={"title": "Python Crash Course", "author": "Eric Matthes", "year": 2019})
show(r)
assert r.json()["id"] == 4
ids = [b["id"] for b in requests.get(f"{BASE}/books").json()]
print("ids:", ids)
assert ids == [2, 3, 4]
print("Task 8 passed")

server.shutdown()

127.0.0.1 - - [11/Oct/2026 20:01:48] "DELETE /books/1 HTTP/1.1" 204 -
127.0.0.1 - - [11/Oct/2026 20:01:48] "GET /books/1 HTTP/1.1" 404 -
127.0.0.1 - - [11/Oct/2026 20:01:48] "DELETE /books/1 HTTP/1.1" 404 -
127.0.0.1 - - [11/Oct/2026 20:01:48] "POST /books HTTP/1.1" 201 -
127.0.0.1 - - [11/Oct/2026 20:01:48] "GET /books HTTP/1.1" 200 -


204 ''
404 {'error': 'Book not found'}
201 {'author': 'Eric Matthes', 'id': 4, 'title': 'Python Crash Course', 'year': 2019}
ids: [2, 3, 4]
Task 8 passed


## Part B - Questions

### Q1. In the lesson the new id was len(users) + 1 
* Describe a sequence of requests that gives two users the same id, and explain why next_id does not have this problem.
---
* Because if we have a list[book1, book2], len(list) = 2 but if we delete book1 then the len(list) = 1 thus the new book would be assigned the id = 2 again
* but next_id keeps increasing after each successful POST so the same id never duplicates

### Q2. Your API answers 200 , 201 and 204 on success.
* Which endpoint uses which code, and why are they different?
---
* 200 OK -> Request succeeded with data returned. -> GET on a resource (Contains the requested payload),
* 201 Created -> New resource was built. ->  POST /books (Usually returns the created object),
* 204 No Content -> Request succeeded without response body. -> DELETE /books/<id> (Strictly empty body)

### Q3. A client sends the body {"year": 2020} to book 2,
* once with PUT and once with PATCH .
What does each one answer, and why?
---
* PUT  -> (status code 400) it Fails Because it doesn't have author and title fields
* PATCH -> (status code 200) it Updates The year Field for the book that has id = 2

### Q4. Why does update_book loop 
* over ("title", "author", "year") instead of calling
book.update(data) ?
---
* Because we are using a python list (List[Dict]) not a JSON Dict
* Python Doesn't have .update() for lists
* also to allow only updating the wanted fields and not the id

### Q5. After a successful DELETE 
* why must the client not call r.json() ? What should it check
instead?
--
* that r.status_code == 204